In [1]:

#PIP Installations
import sys
!{sys.executable} -m pip install pillow tensorflow keras requests numpy roboflow pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 101.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [18]:
#Imports
## System
import os
import sys

## Google Colab
from google.colab import userdata

## HTTP / IO
import urllib.request
import requests
from io import BytesIO

## Datenladen
from roboflow import Roboflow

## Datenverarbeitung
import numpy as np
import pandas as pd
from numpy import asarray

## Bildverarbeitung
from PIL import Image

## Machine Learning
import keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
def convert_yolo_to_cnn_dataset(folder_path: str):
    # Zielordner: /content/uno_cnn/train (oder valid/test)
    # Ordnername wird aus dem letzten Teil des Pfades genommen
    path = f"/content/uno_cnn/{folder_path.split('/')[-1]}"
    os.makedirs(path, exist_ok=True)

    # CSV enthält: filename, class, xmin, ymin, xmax, ymax
    # Jede Zeile = eine Karte im Bild (ein Bild kann mehrere Zeilen haben)
    df = pd.read_csv(f"{folder_path}/_annotations.csv")

    current_file = None
    img = None

    for index, row in df.iterrows():
        # Optimierung: Bild nur neu laden wenn sich der Dateiname ändert
        # Spart Zeit weil mehrere Zeilen oft dasselbe Bild referenzieren
        if row.filename != current_file:
            img = Image.open(f"{folder_path}/{row.filename}")
            current_file = row.filename

        # Karte aus dem Bild ausschneiden anhand der Bounding Box
        # xmin/ymin = obere linke Ecke, xmax/ymax = untere rechte Ecke
        img_crop = img.crop((row["xmin"], row["ymin"], row["xmax"], row["ymax"]))

        # Ordner pro Klasse erstellen (0-14)
        # Keras ImageDataGenerator erwartet diese Struktur:
        # train/0/bild.jpg, train/1/bild.jpg, etc.
        # Der Ordnername wird automatisch zum Label!
        new_path = f"{path}/{row['class']}"
        os.makedirs(new_path, exist_ok=True)

        # Einheitliche Grösse für CNN (alle Bilder müssen gleich gross sein)
        img_resized = img_crop.resize((128, 128))
        img_resized.save(f"{new_path}/{index}.jpg")


In [4]:
import keras

# Sequential = Modell wo Schichten nacheinander gestapelt werden
# Jede Schicht bekommt den Output der vorherigen als Input
model = keras.Sequential()

# Input-Shape definieren: 128x128 Pixel, 3 Farbkanäle (RGB)
# Muss als erste Schicht rein damit das Modell die Bildgrösse kennt
model.add(keras.layers.Input(shape=(128, 128, 3)))

# Conv2D: Gleitet mit 32 verschiedenen 5x5-Fenstern über das Bild
# Jedes Fenster lernt ein anderes Muster zu erkennen (Kanten, Kurven, etc.)
# relu: Negative Werte werden zu 0 -> einfache aber effektive Aktivierung
model.add(keras.layers.Conv2D(filters=32, kernel_size=(5,5), activation="relu"))

# MaxPooling: Verkleinert das Bild auf 1/4 (4x4 -> 1 Wert, der grösste)
# Macht das Modell robuster gegen kleine Verschiebungen im Bild
model.add(keras.layers.MaxPool2D(pool_size=(4,4)))

# Zweite Conv2D-Schicht: Lernt komplexere Muster auf Basis der ersten Schicht
# (z.B. erst Kanten -> dann Ziffernformen)
model.add(keras.layers.Conv2D(filters=32, kernel_size=(5,5), activation="relu"))

# Nochmals verkleinern
model.add(keras.layers.MaxPool2D(pool_size=(4,4)))

# Flatten: Macht aus dem 2D-Bild (6x6x32) eine lange 1D-Liste (1152 Werte)
# Nötig weil Dense-Schichten nur 1D-Input verstehen
model.add(keras.layers.Flatten())

# Dense: Klassische neuronale Schicht mit 30 Nodes
# Kombiniert alle 1152 Werte zu 30 zusammengefassten Merkmalen
model.add(keras.layers.Dense(30, activation="relu"))

# Output-Schicht: 15 Nodes = 15 mögliche Karten-Kategorien
# softmax: Wandelt die 15 Werte in Wahrscheinlichkeiten um (zusammen = 100%)
# z.B. "7" -> 82%, "Skip" -> 11%, "+2" -> 7%
model.add(keras.layers.Dense(15, activation="softmax"))

print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 124, 124, 32)   │         2,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 27, 27, 32)     │        25,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 30)             │        34,590 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │           465 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 63,119 (246.56 KB)

 Trainable params: 63,119 (246.56 KB)

 Non-trainable params: 0 (0.00 B)

None


In [5]:
# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(), # Adaptive Moment Estimation)
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()]
    )

In [13]:
# Load dataset
rf = Roboflow(api_key=userdata.get("KEY_ROBOFLOW"))
project = rf.workspace("joseph-nelson").project("uno-cards")
dataset = project.version(1).download("tensorflow")

loading Roboflow workspace...
loading Roboflow project...


In [15]:
# Convert YOLO dataset to cnn dataset (~ 2-3 min)
convert_yolo_to_cnn_dataset("/content/Uno-Cards-1/test")
convert_yolo_to_cnn_dataset("/content/Uno-Cards-1/train")
convert_yolo_to_cnn_dataset("/content/Uno-Cards-1/valid")


In [16]:
# Generator erstellen (nur Normalisierung)
datagen = ImageDataGenerator(rescale=1./255)

# Train-Daten laden
train_generator = datagen.flow_from_directory(
    "/content/uno_cnn/train",
    target_size=(128, 128),
    batch_size=32,
    class_mode="sparse"
)

# Valid-Daten laden
valid_generator = datagen.flow_from_directory(
    "/content/uno_cnn/valid",
    target_size=(128, 128),
    batch_size=32,
    class_mode="sparse"
)

Found 56655 images belonging to 15 classes.
Found 5394 images belonging to 15 classes.


In [17]:
# Training
model.fit(
    train_generator,
    epochs=10,
    validation_data=valid_generator,
)

Epoch 1/10
1771/1771 ━━━━━━━━━━━━━━━━━━━━ 47s 24ms/step - loss: 0.1655 - sparse_categorical_accuracy: 0.9481 - val_loss: 0.0106 - val_sparse_categorical_accuracy: 0.9968
Epoch 2/10
1771/1771 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - loss: 0.0119 - sparse_categorical_accuracy: 0.9965 - val_loss: 0.0035 - val_sparse_categorical_accuracy: 0.9993
Epoch 3/10
1771/1771 ━━━━━━━━━━━━━━━━━━━━ 36s 20ms/step - loss: 0.0074 - sparse_categorical_accuracy: 0.9979 - val_loss: 0.0073 - val_sparse_categorical_accuracy: 0.9991
Epoch 4/10
1771/1771 ━━━━━━━━━━━━━━━━━━━━ 36s 21ms/step - loss: 0.0059 - sparse_categorical_accuracy: 0.9982 - val_loss: 0.0020 - val_sparse_categorical_accuracy: 0.9996
Epoch 5/10
1771/1771 ━━━━━━━━━━━━━━━━━━━━ 36s 20ms/step - loss: 0.0049 - sparse_categorical_accuracy: 0.9985 - val_loss: 6.6080e-04 - val_sparse_categorical_accuracy: 0.9998
Epoch 6/10
1771/1771 ━━━━━━━━━━━━━━━━━━━━ 39s 22ms/step - loss: 0.0022 - sparse_categorical_accuracy: 0.9994 - val_loss: 0.0029 - val_sparse_categ

In [23]:
# Save model
model.save("uno_model_cnn.keras")

# Keras Modell laden
keras_model = tf.keras.models.load_model('uno_model_cnn.keras')

# Normal konvertieren
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
tflite_model = converter.convert()

# Quantized konvertieren
converter_q = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter_q.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model_quantized = converter_q.convert()

# Speichern
with open('uno_model_cnn.tflite', 'wb') as f:
    f.write(tflite_model)

with open('uno_model_cnn_quantized.tflite', 'wb') as f:
    f.write(tflite_model_quantized)

Saved artifact at '/tmp/tmp4csuqtjl'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 15), dtype=tf.float32, name=None)
Captures:
  139711191635088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139711191634128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139711191637584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139711191630480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139711191632592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139711191637968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139711191635280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139711191638352: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved artifact at '/tmp/tmp20z0n078'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 12